# Lab Workbook No 4 - Part No 1: Making Maps

## Challenge 1
1) Go to this link https://learn.arcgis.com/en/projects/map-and-analyze-the-urban-heat-island-effect/ and follow all the instructions provided. The ETA for this tutorial is 40 mins, but I guess it will take you around 1 hour if you are not familiar with ArcGIS Online.

2) After completing the tutorial, please take a screenshot of your final dashboard and include it in the next markdown cell along with the dashboard link. This is a key step as I will validate the image againts the link you will provide, make sure you share the dasboard with the corresponding group in ArcGIS Online. That is, in the step 11 of the Finish the map section instead of sharing to everyone, you need to clic in Edit group sharing and then find the corresponding group where I will be able to see the dashboard.

1) Completed the Urban Heat Map Tutorial.
Finished map output screenshot:
![Urban_heat_output_map](screenshots/Urban_heat_presented_map.png)
Screenshot of ArcGis Online:
![Arc_Gis_online_screenshot](screenshots/arcgisonline_screenshot.png)
Screenshot of dashboard:
![dashboard_screenshot](screenshots/dashboard_screenshot.png)
3) Link to ArcGIS Online group with the map uploaded click [here](https://arcg.is/1GHqm48), also named:2026 GG4257 - Urban Analytics - ('el221_UoStAndrews',)

### Description
For challenge 1, I followed the [Urban heat tutorial]( https://learn.arcgis.com/en/projects/map-and-analyze-the-urban-heat-island-effect/ ) made by Esri where you look into the factors which are contributing towards the higher urban temperatures in Richmond, Virginia. It is included looking at three main factors: max evening temp, tree canopy cover (%) and impervious surfaces (%). Then taught how to use the zonal stats tool to calculate the max evening temp for each census block and then joining those results to the census block boundaries to let you create a chloropleth map. Then it included organising the three main factors into a group layer which allowed you to explore and configure the pop ups of the different information together when you click on the interactive map you made. After finishing the tutorial, I then made this in a dashboard as well as a more proper map (for good practice). I really enjoyed this exercise as I am not that familiar with Arc online so it was fun to fiddle around with all the settings etc. It was also a good exercise to help understand how spatial analysis and an interactive/visual map can be very helpful in expressing environmental issues (in the case example we did) but also for exploring and finding relationships between relvant factors - in this case it was urban development, vegetation and heat exposure.

## Challenge 2

1) Go to this portal https://www.spatialdata.gov.scot/geonetwork/srv/eng/catalog.search#/home

2) Get the Scottish Index of Multiple Deprivation (SIMD) 2020 dataset and extract the data only for the city of Edinburgh.

3) Create two static choropleth maps (e.g. matplotlib). These maps should represent an attribute you find interesting in the SIMD dataset. Using two different classifier methods, you need to show how the maps appear different even though the data and attributes are the same. Include a clear description of your choice and the difference in the classification method for the attribute chosen (e.g. Plotting histograms with breakpoints(bins)). You can find a complete list of classifiers at https://pysal.org/mapclassify/api.html.

4) Finally, create other two interactive maps (e.g. choropleth_map) - one for Glasgow and one for Edinburgh - to represent the difference in deprivation for both cities. Pick any of the available attributes., what is relevant is you can visualise how different both scottish cities are.

In [1]:
# importing all libraries needed:
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify as mc
import plotly.express as px

#### Steps 1 and 2:Retrieving and inspecting data

In [2]:
# step 1 + 2, couldn't figure out how to filter the dataset to only edinburgh using the website
# So i have decided to use the API url instead as the whole dataset would be too large to try and filter directly here
# using https://spatialdata.gov.scot/geonetwork/doc/api/index.html to help 
# Then used the example from lab 4 to help as well to figure out what the whole api link would be and what parameters I should be keeping
# includes LAName='City of Edinburgh'
url_edinburgh = "https://maps.gov.scot/server/rest/services/ScotGov/PeopleSociety/MapServer/7/query?where=LAName%3D%27City+of+Edinburgh%27&outFields=*&returnGeometry=true&f=geojson"

response = requests.get(url_edinburgh)
response

<Response [200]>

In [3]:
#converting response to json format as usual
data = response.json()
#data

In [4]:
 # converting to a geodataframe next
edi_simd = gpd.GeoDataFrame.from_features(data["features"])
edi_simd.head()

,geometry,objectid,datazone,dzname,laname,sape2017,wape2017,rankv2,quintilev2,decilev2,...,crimerate,crimerank,housenumoc,housenumnc,houseocrat,housencrat,houserank,shape_leng,st_area(shape),st_length(shape)
0,"POLYGON ((-3.18044 55.94393, -3.18067 55.94408...",2167,S01008672,Meadows and Southside - 07,City of Edinburgh,781,663,6331,5,10,...,415,1421.0,205,41,30%,6%,194.0,847.968613,31476.530009,847.968613
1,"POLYGON ((-3.14229 55.90328, -3.14229 55.90358...",2055,S01008560,Hyvots and Gilmerton - 02,City of Edinburgh,678,423,1680,2,3,...,374,1652.0,123,11,18%,2%,1333.0,1604.808594,135538.903747,1604.808594
2,"POLYGON ((-3.13934 55.91860, -3.13945 55.91851...",2064,S01008569,Moredun and Craigour - 01,City of Edinburgh,715,504,26,1,1,...,963,229.0,167,14,26%,2%,428.0,729.377257,29005.565128,729.377257
3,"POLYGON ((-3.10730 55.94418, -3.10639 55.94420...",2211,S01008716,"Bingham, Magdalene and The Christians - 05",City of Edinburgh,568,335,1098,1,2,...,250,2757.0,77,5,14%,1%,2152.0,1267.435447,91629.108107,1267.435447
4,"POLYGON ((-3.26319 55.92193, -3.26306 55.92210...",1978,S01008483,Longstone and Saughton - 03,City of Edinburgh,612,416,1466,2,3,...,381,1609.0,69,5,12%,1%,2951.5,1498.194725,63507.304500,1498.194725


In [ ]:
# Doing some standard checks to see if the API returned the edinburgh data correctly
print(edi_simd.shape)
print(edi_simd.columns)
print(edi_simd.crs)
#the geojson does use lat/long but the CRS was not assigned when making the geodata frame so include:
edi_simd = edi_simd.set_crs('epsg:4326') #so that we can map using plotly later
print(edi_simd.crs) #checking it worked
print(edi_simd['laname'].unique())

# Then checking the distribution of the chosen income deprivation attribute- 'incrate'
edi_simd['hlthalcsr'].describe()

#### Step 3: Create two static choropleth maps

In [6]:
# need to calculate the classifications to see the class boundaries and see how many edinburgh data zones fall into each class
# wanting to compare the natural breaks and equal interval examples from lab 4 but with standardised ratio for alcohol-related hospital stays in edi
#number of classes for classification = 5
num_classes = 5

#using the natural breaks method which is in lab 4 workbook:
# also using http://nbviewer.jupyter.org/github/pysal/mapclassify/blob/main/notebooks/03_choropleth.ipynb to help

#Natural break first:
classifier_nb = mc.NaturalBreaks(edi_simd['hlthalcsr'], k=num_classes)
print(classifier_nb)
print(min(classifier_nb.bins), max(classifier_nb.bins))
print(classifier_nb.bins) 

# Then equal interval classification, also taken from lab 4 workbook
classifier_ei = mc.EqualInterval(edi_simd['hlthalcsr'], k=num_classes)
print(classifier_ei)
print(min(classifier_ei.bins), max(classifier_ei.bins))
print(classifier_ei.bins)

C:\Users\elisa\miniforge3\envs\ua\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


NaturalBreaks

    Interval       Count
------------------------
[  0.00,  46.00] |   241
( 46.00, 102.00] |   181
(102.00, 179.00] |   108
(179.00, 285.00] |    51
(285.00, 518.00] |    16
46.0 518.0
[ 46. 102. 179. 285. 518.]
EqualInterval

    Interval       Count
------------------------
[  0.00, 103.60] |   422
(103.60, 207.20] |   129
(207.20, 310.80] |    34
(310.80, 414.40] |     8
(414.40, 518.00] |     4
103.6 518.0
[103.6 207.2 310.8 414.4 518. ]


In [ ]:
# Thought it would be good practise to comparing the histpgrams comparing the endinburgh income deprivation rates with the class boundaries from each method
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# First is the natural breaks method:
sns.histplot(data=edi_simd, x="hlthalcsr", ax=axes[0], kde=True, bins=20)
axes[0].axvline(classifier_nb.bins[0], color='red', linestyle='dashed', linewidth=2, label='Natural Breaks')
for bin_value in classifier_nb.bins:
    axes[0].axvline(bin_value, color='red', linestyle='dashed', linewidth=2)
axes[0].set_title("EDI alcohol related hospital stays histogram with Natural Breaks")
axes[0].legend()

# Second is the equal interval method
# done with help of reference: http://nbviewer.jupyter.org/github/pysal/mapclassify/blob/main/notebooks/03_choropleth.ipynb
sns.histplot(data=edi_simd, x="hlthalcsr", ax=axes[1], kde=True, bins=20)
axes[1].axvline(classifier_ei.bins[0], color='blue', linestyle='dashed', linewidth=2, label='Equal Intervals')
for bin_value in classifier_ei.bins:
    axes[1].axvline(bin_value, color='blue', linestyle='dashed', linewidth=2)
axes[1].set_title("EDI alcohol related hospital stays histogram with Equal Intervals")
axes[1].legend()

#changing the x axis labels to be better represent the data
axes[0].set_xlabel("Standardised ratio")
axes[1].set_xlabel("Standardised ratio")

# then adjusting the plot
plt.tight_layout()
plt.show()

![edinburgh_histograms](screenshots/edinburgh_histograms.png)

In [ ]:
#Making the graphs for the natural breaks method and the equal interval method and presenting them side by side using code from lab 4
#Both maps use the same dataset+colourscheme so the methods can be compared easily but they do use different classification boundaries as calculautes above
fig, axs = plt.subplots(1, 2, figsize=(18, 8))

#Frist is the Natural Breaks method:
edi_simd.plot(column='hlthalcsr', ax=axs[0],
         legend=True, cmap='viridis',
         scheme='UserDefined',
         classification_kwds={'bins': classifier_nb.bins}
        )

axs[0].set_title("EDI - alcohol related hospital stays with Natural Breaks")

edi_simd.plot(column='hlthalcsr', ax=axs[1],
         legend=True, cmap='viridis',
         scheme='UserDefined',
         classification_kwds={'bins': classifier_ei.bins})

axs[1].set_title("EDI - alcohol related hospital stays with Equal Intervals")

plt.tight_layout() 
plt.show()

![edinburgh_static](screenshots/edinburgh_static_maps.png)

#### Step 4: Interactive maps

In [ ]:
# Making figures interactive using Plotly:
# Need to create a column with the values from the classifier object then using that new column to plot the map
#Starting with edinburgh:
#Decided to use the existing quintile2 field instead of recalculting the classes for edi
fig = px.choropleth_map(edi_simd,
                           geojson=edi_simd.geometry,
                           locations=edi_simd.index,
                           color="quintilev2",
                           color_continuous_scale="viridis",
                           range_color= (1, 5),
                           opacity=0.5, center={"lat": 55.9533, "lon": -3.1883}, #found the centre coords on google
                           map_style="carto-positron",
                           zoom=9.5)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

![edinburgh_interactive](screenshots/edinburgh_interactive_map.png)

In [ ]:
# Glasgow data for interactive map:
#First need to load the glasgow simd 2020 shapefile from lab 4
shapefile_path = 'data/SIMD_2020_GlasgowCity.shp'
glasgow_simd = gpd.read_file(shapefile_path)

#doing some wuick general checks:
print(glasgow_simd.head())
print(glasgow_simd.columns)
print(glasgow_simd.shape)
print(glasgow_simd.crs)
print(glasgow_simd['LAName'].unique())
glasgow_simd = glasgow_simd.to_crs('epsg:4326') #just in case

In [ ]:
#Making the glasgow interactive simd map using the same national SIMD 2020 quintiles
#and keeping the colour scale and classification the same as the edinburgh map for better comparison

fig_2 = px.choropleth_map(glasgow_simd,
                           geojson=glasgow_simd.geometry,
                           locations=glasgow_simd.index,
                           color="Quintilev2",
                           color_continuous_scale="viridis",
                           range_color= (1, 5),
                           opacity=0.5,
                           center={"lat": 55.866193, "lon": -4.258246},
                           map_style="carto-positron",
                           zoom=9.5)
fig_2.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig_2.show()

![glasgow_interactive](screenshots/glasgow_interactive_map.png)

### Description 
For challenge 2, used the Scottish Index of Multiple Derpivation (2020) data as requested with the aim of investigating how different classification methods affect how deprivation across Edinburgh (and Glasgow) differs visually with this. I initially struggled with finding and downloading the data on the website because I could not figure out how to filter the dataset so that I wasn't downloading a very large file. However, I couldn't figure it out so I opted for using their API url and API documentation help page to instead directly filter the data through changing the parameters in the url instead (like what was done in lab 4 to get Edinburgh's 597 data zones).
After doing the standard checks for the data and assigning the crs to allow me to use plotly later, i chose to use 'hlthalcsr' which represents the data for the standardised ratio for alcohol-related hospital stays as I thought this could be interesting to look into in terms of it beig a health-related deprivation index.
Following from lab 4, i used natural breaks and equal interval and keeping the 5 classes and the same colour scheme for both maps. The side by side histogram and static chloropleth figured for edinburgh are important because they show how changing the class boundaries do alter the apparant spatial variation of the data despite the base data being the same. Visually looking at the figures, you can see that Natural breaks put more data zones in the intermediate classes, while the equal intervals out more observations in the lower classes.
Then finally, I used the glasgow shapefile and edinburgh api data to create the two required interactive plotly maps using the existing national SIMD quintiles (note that I didn't include as much analysis of the glasgow data since we already went through that in the lab example so i thought that might be a bit repetative/uneccesary to include here).
Having the two final interactive maps is important because it allows the 2 cities to be compared with consistent factors including a consistent deprivation classification - which also showed how both the chosen attribute and classification method can easily influence or mislead the interpretation of spatial patterns depending on your method.
